In [1]:
# Import Required Libaries

import pandas as pd
import numpy as np

In [2]:
# Load Dataset
df = pd.read_csv("used_cars_clean.csv")
df.head()

,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice,Posted_Year,Posted_Month,Posted_Month_Name
0,Honda,City,2001,23,98000.0,Manual,second,Petrol,2024-11-01,"Honda City v teck in mint condition, valid gen...",195000,2024,11,November
1,Toyota,Innova,2009,15,190000.0,Manual,second,Diesel,2024-07-01,"Toyota Innova 2.5 G (Diesel) 7 Seater, 2009, D...",375000,2024,7,July
2,Volkswagen,VentoTest,2010,14,77246.0,Manual,first,Diesel,2024-11-01,"Volkswagen Vento 2010-2013 Diesel Breeze, 2010...",184999,2024,11,November
3,Maruti Suzuki,Swift,2017,7,83500.0,Manual,second,Diesel,2024-11-01,Maruti Suzuki Swift 2017 Diesel Good Condition,565000,2024,11,November
4,Maruti Suzuki,Baleno,2019,5,45000.0,Automatic,first,Petrol,2024-11-01,"Maruti Suzuki Baleno Alpha CVT, 2019, Petrol",685000,2024,11,November


In [6]:
# # 3) Basic Dataset Overview

print("Dataset Overview")
print("-" * 70)

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\nNumerical Features")
display(df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical Features")
display(df.select_dtypes(include="object").columns.tolist())

print("\n")

df.info()

Dataset Overview
----------------------------------------------------------------------
Rows    : 13982
Columns : 14

Numerical Features


['Year', 'Age', 'kmDriven', 'AskPrice', 'Posted_Year', 'Posted_Month']


Categorical Features


['Brand',
 'model',
 'Transmission',
 'Owner',
 'FuelType',
 'PostedDate',
 'AdditionInfo',
 'Posted_Month_Name']



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13982 entries, 0 to 13981
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Brand              13982 non-null  object 
 1   model              13982 non-null  object 
 2   Year               13982 non-null  int64  
 3   Age                13982 non-null  int64  
 4   kmDriven           13895 non-null  float64
 5   Transmission       13982 non-null  object 
 6   Owner              13982 non-null  object 
 7   FuelType           13982 non-null  object 
 8   PostedDate         13982 non-null  object 
 9   AdditionInfo       13982 non-null  object 
 10  AskPrice           13982 non-null  int64  
 11  Posted_Year        13982 non-null  int64  
 12  Posted_Month       13982 non-null  int64  
 13  Posted_Month_Name  13982 non-null  object 
dtypes: float64(1), int64(5), object(8)
memory usage: 1.5+ MB


In [3]:
# Inspect AdditionInfo

print("Number of Unique Values:")
print(df["AdditionInfo"].nunique())

print("\nRandom Samples:\n")

print(df["AdditionInfo"].sample(20, random_state=42).to_string(index=False))

Number of Unique Values:
10673

Random Samples:

     Mercedes-Benz CLS-Class 350 CDI, 2017, Diesel
          Hyundai Getz Prime 1.3 GLS, 2008, Diesel
                Honda Amaze V Diesel, 2019, Diesel
             Hyundai Santro Xing GLS, 2011, Petrol
    Hyundai i20 2014 CNG & Hybrids 70000 Km Driven
    Audi Q7 3.0 Premium Plus 55 TFSI, 2022, Petrol
   Tata Safari Storme VX Varicor 400, 2017, Diesel
          Kam nhi hoga please kam wale sms na kare
Hyundai Verna Fluidic 1.6 VTVT SX Automatic, 20...
Maruti Suzuki Alto 800 2013 Petrol 80000 Km Driven
      Mercedes-Benz CLA 2016 Diesel Good Condition
             All cars of different model available
Ford Ecosport 1.2 Titanium Plus Sports, 2019, C...
  Hyundai Grand i10 2013-2016 Sportz, 2016, Petrol
Maruti Suzuki Celerio X 2018 CNG & Hybrids 5800...
     Maruti Suzuki Vitara Brezza VDi, 2018, Diesel
Maruti Suzuki Swift Dzire VXI(O) AMT, 2012, Petrol
    Hyundai Creta 1.6 CRDi SX Option, 2016, Diesel
    BMW 5 Series 2.0 530i Sport L

**Interpretation:**
- The **AdditionInfo** column contains free-form textual descriptions with high variability and inconsistent formatting. While some records include useful information such as trim level or engine specifications, many entries contain redundant or promotional text that is difficult to standardize. Since the final application is designed to predict prices from structured user inputs, incorporating this column would complicate both preprocessing and deployment.

- To ensure a simple, user-friendly, and production-ready prediction system, only structured features that users can reliably provide are included in the final model. Excluding AdditionInfo improves the consistency of user inputs, simplifies the preprocessing pipeline, and makes the deployed Flask application easier to use while maintaining strong predictive performance.

In [7]:
# Step 1 - Drop Unnecessary Features

columns_to_drop = [
    "AdditionInfo",
    "PostedDate",
    "Posted_Year",
    "Posted_Month_Name"
]

df.drop(columns=columns_to_drop, inplace=True)

print("Remaining Columns:")
print(df.columns.tolist())

Remaining Columns:
['Brand', 'model', 'Year', 'Age', 'kmDriven', 'Transmission', 'Owner', 'FuelType', 'AskPrice', 'Posted_Month']


In [9]:
# Drop Age

df.drop(columns="Age", inplace=True)
print(df.columns)

Index(['Brand', 'model', 'Year', 'kmDriven', 'Transmission', 'Owner',
       'FuelType', 'AskPrice', 'Posted_Month'],
      dtype='object')


In [10]:
# Step 2- Create km_per_year

CURRENT_YEAR = 2026

vehicle_age = CURRENT_YEAR - df["Year"]

# Avoid division by zero for new vehicles
vehicle_age = vehicle_age.replace(0, 1)

df["km_per_year"] = df["kmDriven"] / vehicle_age

print(df[["Year", "kmDriven", "km_per_year"]].head())

   Year  kmDriven   km_per_year
0  2001   98000.0   3920.000000
1  2009  190000.0  11176.470588
2  2010   77246.0   4827.875000
3  2017   83500.0   9277.777778
4  2019   45000.0   6428.571429


In [11]:
# Step 3- Inspect the New Feature

# Summary Statistics
print(df["km_per_year"].describe())

# Missing Values
print("\nMissing Values:", df["km_per_year"].isnull().sum())

count     13895.000000
mean       7940.616465
std        7333.607335
min           0.000000
25%        5000.000000
50%        6800.000000
75%        9375.000000
max      225375.000000
Name: km_per_year, dtype: float64

Missing Values: 87


In [12]:
# Drop Posted_Month

df.drop(columns=["Posted_Month"], inplace=True)

print("Final Features:")
print(df.columns.tolist())

print("\nDataset Shape:", df.shape)

Final Features:
['Brand', 'model', 'Year', 'kmDriven', 'Transmission', 'Owner', 'FuelType', 'AskPrice', 'km_per_year']

Dataset Shape: (13982, 9)


In [13]:
# Save the Feature Engineered Dataset

df.to_csv("feature_engineered_dataset.csv", index=False)
print("Feature Engineered Dataset Saved Successfully!")

Feature Engineered Dataset Saved Successfully!
